In [25]:
from langgraph.graph import StateGraph , START , END
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict ,Literal

In [2]:
class QuadraticInput(TypedDict):
    a: float
    b: float
    c: float

    equation: str
    discriminant: float
    result : str

In [8]:
def show_equation(state: QuadraticInput) -> QuadraticInput:

    equa = f"{state['a']}x^2 + {state['b']}x + {state['c']} = 0"
   
    return {'equation': equa}


In [26]:
def calculate_discriminant(state: QuadraticInput)-> QuadraticInput:

    discr = state['b']**2 - 4*state['a']*state['c']
    return {'discriminant': discr}

def real_root(state: QuadraticInput) -> QuadraticInput:

    root1 = (-state['b'] + (state['discriminant'])**0.5) / (2*state['a'])
    root2 = (-state['b'] - (state['discriminant'])**0.5) / (2*state['a'])

    result = f"The roots are real and different. The roots are {root1} and {root2}"
    return {'result': result}

def repeated_root(state: QuadraticInput) -> QuadraticInput:

    root = -state['b'] / (2*state['a'])
    result = f"The roots are real and same. The root is {root}"
    return {'result': result}

def no_real_root(state: QuadraticInput) -> QuadraticInput:

    result = "The roots are complex and different."
    return {'result': result}

def check_condition(state: QuadraticInput ) -> Literal['real_root', 'repeated_root', 'no_real_root']:

    if state['discriminant'] > 0:
        return 'real_root'
    elif state['discriminant'] == 0:
        return 'repeated_root'
    else:
        return 'no_real_root'


In [28]:
graph = StateGraph(QuadraticInput)

graph.add_node("show_equation",show_equation)
graph.add_node("calculate_discriminant",calculate_discriminant)
graph.add_node('real_root',real_root)
graph.add_node('repeated_root',repeated_root)
graph.add_node('no_real_root',no_real_root)

graph.add_edge(START , "show_equation")
graph.add_edge("show_equation" , "calculate_discriminant")
graph.add_conditional_edges("calculate_discriminant", check_condition )
graph.add_edge("real_root", END)
graph.add_edge("repeated_root", END)
graph.add_edge("no_real_root", END)

workflow = graph.compile()

# graph = StateGraph(QuadraticInput)

# graph.add_node("show_equation", show_equation)
# graph.add_node("calculate_discriminant", calculate_discriminant)

# graph.add_edge(START, "show_equation")
# graph.add_edge("show_equation", "calculate_discriminant")
# graph.add_edge("calculate_discriminant", END)

# workflow = graph.compile()

In [29]:
intial_state = {
    'a': 1,
    'b': -3,
    'c': 2,
}
workflow.invoke(intial_state)

{'a': 1,
 'b': -3,
 'c': 2,
 'equation': '1x^2 + -3x + 2 = 0',
 'discriminant': 1,
 'result': 'The roots are real and different. The roots are 2.0 and 1.0'}